# Task 005b — Value score (deterministic weighted formula)

Computes a 0-1 `value_score` and a categorical `value_tier` (`bronze`/`silver`/`gold`/`platinum`) for every eligible client (train + test + holdout = 4,614 rows).

The score is a deterministic weighted sum of six percentile-ranked components — **not** a trained model. Percentile ranks are computed across the full eligible universe so every client (including holdout) lands on the same scale. This is not data leakage because no fitting occurs.

**Outputs**
- `data/processed/features_value.parquet` — one row per client, 6 components + `value_score` + `value_tier`
- `data/models/value_weights.json` — weights dump + credit-rating convention note

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import warnings

import numpy as np
import pandas as pd

import config

PROC   = config.PROCESSED_DATA_DIR
MODELS = config.MODELS_DIR
MODELS.mkdir(parents=True, exist_ok=True)

## 1. Load inputs

Inner-merge `features_churn.parquet` with `clients_eligible.parquet` on `IDENTIFIKATOR_KLIJENTA` to bring `KREDITNI_RATING` in from the raw client table (in case the engineered feature has been further processed or clipped). Features already expose `receives_primary_income_at_bank` as a clean 0/1 int, `total_balance_avg`, `tx_amount_total`, `months_since_first_relationship`, and all `has_*` flags.

In [3]:
features  = pd.read_parquet(PROC / 'features_churn.parquet')
clients   = pd.read_parquet(PROC / 'clients_eligible.parquet')

# Use raw KREDITNI_RATING from clients (not the coerced features copy) to
# make the credit-rating convention decision explicit on the source column.
df = features.merge(
    clients[['IDENTIFIKATOR_KLIJENTA', 'KREDITNI_RATING']],
    on='IDENTIFIKATOR_KLIJENTA', how='inner', suffixes=('', '_raw'),
)

print(f'features shape : {features.shape}')
print(f'clients shape  : {clients.shape}')
print(f'merged shape   : {df.shape}')
assert df['IDENTIFIKATOR_KLIJENTA'].is_unique, 'Duplicate client IDs after merge!'

features shape : (4614, 83)
clients shape  : (4614, 39)
merged shape   : (4614, 84)


## 2. Compute 6 component scores across all eligible clients

All ranks use `method='average'` (the default) with `pct=True` and `na_option='bottom'` so that missing values fall to the lowest percentile band rather than contaminating the ranking.

### Credit-rating convention

The raw `KREDITNI_RATING` in this dataset uses the Croatian convention where **lower numeric values are better** (1 = best, higher = worse). We therefore rank the negated rating — i.e. the client with `KREDITNI_RATING = 1` gets a near-1.0 `credit_rating_score`. Missing ratings are imputed with the median of non-null ratings before ranking (falling back to 0.5 if the column is entirely empty).

In [4]:
# 2.1 balance_score — percentile rank of total_balance_avg
df['balance_score'] = df['total_balance_avg'].rank(pct=True, na_option='bottom')

# 2.2 product_depth_score — weighted product count then percentile rank
product_depth_raw = (
    3 * (df['has_mortgage'] + df['has_loan'] + df['has_investment'])
    + 2 * df['has_credit_card']
    + (df['has_checking'] + df['has_savings'])
)
df['product_depth_score'] = product_depth_raw.rank(pct=True, na_option='bottom')

# 2.3 revenue_score — rank tx_amount_total, average with product_depth_score, re-rank
tx_amount_rank = df['tx_amount_total'].rank(pct=True, na_option='bottom')
df['revenue_score'] = ((tx_amount_rank + df['product_depth_score']) / 2.0).rank(
    pct=True, na_option='bottom',
)

# 2.4 tenure_score — upstream date-parse issue leaves months_since_first_relationship
# entirely NaN (task 003 carries a latent bug). Rank on an all-NaN column yields
# NaN for every row; we instead set a uniform 0.5 so the weighting math still runs
# and the situation is visible in logs.
mfr = df['months_since_first_relationship']
if mfr.notna().any():
    df['tenure_score'] = mfr.rank(pct=True, na_option='bottom')
else:
    warnings.warn(
        'months_since_first_relationship is 100% NaN — tenure_score degenerates to 0.5. '
        'Upstream fix needed in notebooks/03_features_churn.ipynb date parsing.',
        stacklevel=2,
    )
    df['tenure_score'] = 0.5

# 2.5 primary_bank_score — feature column is already int 0/1
df['primary_bank_score'] = df['receives_primary_income_at_bank'].astype(float).fillna(0.0)

# 2.6 credit_rating_score — lower raw value = better rating, so rank the negation
cr = pd.to_numeric(df['KREDITNI_RATING'], errors='coerce')
if cr.notna().any():
    cr_imputed = cr.fillna(cr.median())
    df['credit_rating_score'] = (-cr_imputed).rank(pct=True, na_option='bottom')
else:
    warnings.warn('KREDITNI_RATING is entirely NaN — credit_rating_score set to 0.5 for all rows.', stacklevel=2)
    df['credit_rating_score'] = 0.5

component_cols = list(config.VALUE_WEIGHTS.keys())
print('Component score summary:')
print(df[component_cols].describe().T[['min', '25%', '50%', '75%', 'max', 'mean']].round(4))

Component score summary:
                        min     25%     50%     75%     max    mean
balance_score        0.0002  0.2502  0.5001  0.8557  0.8557  0.5001
revenue_score        0.0002  0.2502  0.5001  0.7500  1.0000  0.5001
product_depth_score  0.1467  0.1467  0.4598  0.7607  0.9985  0.5001
tenure_score         0.5000  0.5000  0.5000  0.5000  0.5000  0.5000
primary_bank_score   0.0000  0.0000  0.0000  1.0000  1.0000  0.4469
credit_rating_score  0.0009  0.5354  0.5354  0.5354  0.9998  0.5001


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: months_since_first_relationship is 100% NaN — tenure_score degenerates to 0.5. Upstream fix needed in notebooks/03_features_churn.ipynb date parsing.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 3. Weighted sum → `value_score`

In [5]:
assert set(component_cols) == set(config.VALUE_WEIGHTS.keys()), 'component_cols vs weights mismatch'

weighted_sum = sum(df[c] * w for c, w in config.VALUE_WEIGHTS.items())
df['value_score'] = weighted_sum.clip(0.0, 1.0)

print('value_score summary:')
print(df['value_score'].describe().round(4))

value_score summary:
count    4614.0000
mean        0.4948
std         0.1428
min         0.1163
25%         0.3914
50%         0.4722
75%         0.5987
max         0.8593
Name: value_score, dtype: float64


## 4. Assign value tiers

Bin edges are taken from `config.VALUE_TIERS`. The upper edge is bumped by `1e-9` so that a score of exactly 1.0 lands in `platinum` (`pd.cut` default is right-inclusive, but the edge must strictly exceed the max value to avoid NaN on the right tail when `include_lowest=True`).

In [6]:
bins   = [0.0, 0.40, 0.70, 0.90, 1.0 + 1e-9]
labels = ['bronze', 'silver', 'gold', 'platinum']

df['value_tier'] = pd.cut(
    df['value_score'], bins=bins, labels=labels, include_lowest=True,
).astype(str)

print('Tier distribution:')
print(df['value_tier'].value_counts().reindex(labels).to_string())

Tier distribution:
value_tier
bronze      1344.0
silver      2833.0
gold         437.0
platinum       NaN


## 5. Save outputs

In [7]:
out_cols = ['IDENTIFIKATOR_KLIJENTA'] + component_cols + ['value_score', 'value_tier']
out_df   = df[out_cols].copy()

out_parquet = PROC / 'features_value.parquet'
out_df.to_parquet(out_parquet, index=False)
print(f'Saved: {out_parquet}   shape={out_df.shape}')

out_weights = MODELS / 'value_weights.json'
weights_payload = {
    'weights': config.VALUE_WEIGHTS,
    'tiers':   {k: list(v) for k, v in config.VALUE_TIERS.items()},
    'notes': {
        'credit_rating_convention': (
            'Lower raw KREDITNI_RATING = better credit. '
            'credit_rating_score is pct-rank of (-KREDITNI_RATING) '
            'with median imputation for NaNs.'
        ),
        'tenure_caveat': (
            'months_since_first_relationship is 100% NaN upstream; '
            'tenure_score is set uniformly to 0.5 until the date parser is fixed.'
        ),
    },
}
with open(out_weights, 'w') as f:
    json.dump(weights_payload, f, indent=2)
print(f'Saved: {out_weights}')

Saved: /Users/Max/Desktop/Fintech-hackathon/data/processed/features_value.parquet   shape=(4614, 9)
Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/value_weights.json


## 6. Sanity checks

In [8]:
print('=' * 60)
print('ACCEPTANCE CRITERIA CHECKS')
print('=' * 60)

check = pd.read_parquet(out_parquet)

expected_rows = len(clients)
assert len(check) == expected_rows, f'Row count {len(check)} != {expected_rows}'
print(f'[PASS] 1. Row count: {len(check):,} == eligible clients ({expected_rows:,})')

assert check['IDENTIFIKATOR_KLIJENTA'].is_unique, 'Duplicate client IDs!'
assert check['IDENTIFIKATOR_KLIJENTA'].nunique() == expected_rows
print(f'[PASS] 2. IDENTIFIKATOR_KLIJENTA unique across {expected_rows:,} rows')

for c in component_cols:
    assert check[c].between(0.0, 1.0).all(), f'{c} has values outside [0, 1]'
print(f'[PASS] 3. All 6 component scores in [0, 1]')

assert check['value_score'].between(0.0, 1.0).all(), 'value_score out of [0, 1]'
print(f'[PASS] 4. value_score in [0, 1]   (min={check["value_score"].min():.4f}, max={check["value_score"].max():.4f})')

valid_tiers = set(labels)
assert set(check['value_tier'].unique()).issubset(valid_tiers), (
    f'Unexpected tier values: {set(check["value_tier"].unique()) - valid_tiers}'
)
print(f'[PASS] 5. value_tier only contains {sorted(valid_tiers)}')

nan_cols = [c for c in out_cols if check[c].isna().any()]
assert not nan_cols, f'NaN found in output columns: {nan_cols}'
print(f'[PASS] 6. No NaN in any output column')

print()
print('Tier distribution:')
print(check['value_tier'].value_counts().reindex(labels).to_string())
print()
print('value_score five-number summary:')
print(check['value_score'].describe(percentiles=[0.25, 0.5, 0.75]).round(4).to_string())
print()
print('ALL CHECKS PASSED.')

ACCEPTANCE CRITERIA CHECKS
[PASS] 1. Row count: 4,614 == eligible clients (4,614)
[PASS] 2. IDENTIFIKATOR_KLIJENTA unique across 4,614 rows
[PASS] 3. All 6 component scores in [0, 1]
[PASS] 4. value_score in [0, 1]   (min=0.1163, max=0.8593)
[PASS] 5. value_tier only contains ['bronze', 'gold', 'platinum', 'silver']
[PASS] 6. No NaN in any output column

Tier distribution:
value_tier
bronze      1344.0
silver      2833.0
gold         437.0
platinum       NaN

value_score five-number summary:
count    4614.0000
mean        0.4948
std         0.1428
min         0.1163
25%         0.3914
50%         0.4722
75%         0.5987
max         0.8593

ALL CHECKS PASSED.
